# 00 · Setup and split

**Purpose.** Check the environment, create the experiment database, download CIFAR-10, build the stratified member / non-member / validation / shadow split and sample the shadow datasets.

**Inputs.** `config/experiment.yaml`; internet access for the CIFAR-10 archive (163 MB).

**Outputs.** `results/mia.sqlite` (schema + 60,000 `samples` rows), `data/raw/`, `data/splits/split_seed0.npz`.

**Syllabus topics.** Fundamentals of image formation and colour spaces (RGB→BGR conversion happens once, in `mia.data`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

## Environment check
`torch.cuda.is_available() == False` is acceptable: YOLO then trains on the CPU (roughly 10× slower).

In [ ]:
import platform

import cv2, numpy as np, pandas as pd, pywt, sklearn, torch, ultralytics

print(f"Python {platform.python_version()} on {platform.system()} {platform.release()}")
for name, mod in [("opencv", cv2), ("numpy", np), ("pandas", pd), ("pywavelets", pywt), ("scikit-learn", sklearn),
                  ("torch", torch), ("ultralytics", ultralytics)]:
    print(f"{name:>13}: {mod.__version__}")
print("CUDA available:", torch.cuda.is_available(),
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")

## Database, download and image pool

In [ ]:
from mia import db
from mia.config import DB_PATH, RAW_DIR, SPLIT_PATH, load_experiment
from mia.data import CLASS_NAMES, download_cifar10, load_pool, load_split, make_shadow_sets, make_split, save_split

exp = load_experiment()
conn = db.connect(DB_PATH)
download_cifar10(RAW_DIR)
images, labels, source = load_pool(RAW_DIR)
print("pool:", images.shape, images.dtype, "| labels:", np.bincount(labels))

## Stratified split (seed 0) and shadow sets
Per class: 1,000 member, 1,000 non-member, 500 validation, 3,500 shadow. Shadow *k* draws 1,000 IN + 1,000 OUT per class from the shadow pool (`default_rng(1000 + k)`) and a 200-per-class validation set from the remainder (`default_rng(2000 + k)`). Shadow IN/OUT rows go into `shadow_membership` when each shadow model is trained (the table references `models`).

In [ ]:
split = make_split(labels, exp)
n_sets = max(exp.classical_k, exp.yolo_k)
shadow_sets = [make_shadow_sets(split["shadow"], labels, k, exp) for k in range(n_sets)]
if SPLIT_PATH.exists():
    saved = load_split(SPLIT_PATH)
    assert all(np.array_equal(saved[p], split[p]) for p in split), "saved split differs from a fresh one (seed changed?)"
    print("split file exists and matches:", SPLIT_PATH)
else:
    save_split(split, shadow_sets, SPLIT_PATH)
    print("split saved:", SPLIT_PATH)

for k, s in enumerate(shadow_sets):
    assert not set(s["in_idx"]) & set(s["out_idx"]) and not set(s["val_idx"]) & (set(s["in_idx"]) | set(s["out_idx"]))
    assert set(np.concatenate(list(s.values()))) <= set(split["shadow"]), f"shadow {k} leaves the shadow pool"

partition = np.empty(60000, dtype=object)
for p, idx in split.items():
    partition[idx] = p
if conn.execute("SELECT COUNT(*) FROM samples").fetchone()[0] != 60000:
    with conn:
        conn.execute("DELETE FROM samples")
        db.insert_sample_rows(conn, [(i, int(labels[i]), source[i], partition[i]) for i in range(60000)])
print("samples rows:", conn.execute("SELECT COUNT(*) FROM samples").fetchone()[0])

## Partition sizes per class

In [ ]:
sizes = db.query_df(conn, "SELECT label, partition, COUNT(*) AS n FROM samples GROUP BY label, partition")
table = sizes.pivot(index="label", columns="partition", values="n")[["member", "nonmember", "val", "shadow"]]
table.index = [CLASS_NAMES[i] for i in table.index]
table.loc["total"] = table.sum()
table

## A 10 × 10 grid (one row per class)

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
fig, axes = plt.subplots(10, 10, figsize=(9, 9.5))
for c in range(10):
    for j, i in enumerate(rng.choice(np.flatnonzero(labels == c), 10, replace=False)):
        axes[c, j].imshow(images[i][..., ::-1], interpolation="nearest")
        axes[c, j].axis("off")
    axes[c, 0].set_title(CLASS_NAMES[c], fontsize=8, loc="left")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Samples: {len(labels)} | member {len(split['member'])} | nonmember {len(split['nonmember'])} | "
      f"val {len(split['val'])} | shadow pool {len(split['shadow'])} | shadow sets {n_sets}")

## Results
The cell above prints the final partition sizes; the table shows they are balanced per class. The split file and the `samples` table are the fixed basis for every later notebook.